In [100]:
import pandas as pd
from qiskit.quantum_info import SparsePauliOp, Statevector, Pauli
from qiskit.circuit.library import PauliEvolutionGate
import numpy as np
from scipy.linalg import expm
import multiprocessing as mp
from dataclasses import dataclass
import numpy as np
from te_pai import pai, sampling
from qulacs import QuantumCircuit, Observable, QuantumState
from qulacs.gate import X, PauliRotation, DenseMatrix
from scipy.sparse.linalg import expm_multiply

In [ ]:
# Import Hamiltonian data
# This is rightmost is qubit 0 (LSB)
df = pd.read_csv("ES_H4_linear_R1.2_sto-6g.csv")
coeffs = df["coef"].astype(float).tolist()
paulis = df["label"].tolist()
nq = len(paulis[0]) if paulis else 0

In [ ]:
# initial Hartree-Fock state
psi0 = QuantumState(nq)
psi0.set_computational_basis(int("11110000", 2))
T = 1.0

In [128]:
def label_to_qulacs_str(label: str) -> str:
    terms = []
    for q in range(nq):                 
        p = label[nq-1-q]               
        if p != "I":
            terms.append(f"{p} {q}")
    return " ".join(terms)

H = Observable(nq)
for lab, c in zip(paulis, coeffs):
    H.add_operator(c, label_to_qulacs_str(lab))

In [ ]:
Hmat = H.get_matrix()
psiT = expm_multiply((-1j*T) * Hmat, psi0.get_vector()) 

def z_expect_from_vec(vec, q):
    prob = np.abs(vec)**2
    bits = (np.arange(prob.size) >> q) & 1  
    return float(np.sum((1 - 2*bits) * prob))

def site_occupations_from_vec(vec, n_qubits):
    occs = []
    for i in range(n_qubits // 2):
        q_up, q_dn = 2*i, 2*i+1
        n_up = 0.5 * (1 - z_expect_from_vec(vec, q_up))
        n_dn = 0.5 * (1 - z_expect_from_vec(vec, q_dn))
        occs.append(n_up + n_dn)
    return occs

occs = site_occupations_from_vec(psiT, nq)
for i, ni in enumerate(occs):
    print(f"site {i}: n_i = {ni:.6f}")

site 0: n_i = 0.072013
site 1: n_i = 0.093482
site 2: n_i = 1.909448
site 3: n_i = 1.925057


In [132]:
# ==== 入力 ====
csv_path = "ES_H4_linear_R1.2_sto-6g.csv"
T = 1.0
hf_bitstring = "11110000"  # 右端が qubit0 (LSB)

# ==== Hamiltonian (SparsePauliOp) ====
df = pd.read_csv(csv_path)
labels = df["label"].astype(str).tolist()
coeffs = df["coef"].astype(float).tolist()
nq = len(labels[0])

H = SparsePauliOp.from_list(list(zip(labels, coeffs)))

# ==== 初期状態 ====
# Qiskitは左端=最高位 qubit なので bitstring を逆順にする
sv0 = Statevector.from_label(hf_bitstring[::-1])

# ==== 厳密ユニタリ U = exp(-i H T) ====
U = PauliEvolutionGate(H, time=T, synthesis=MatrixExponential())
svT = sv0.evolve(U)

# ==== Z期待値とサイト占有数 ====
def z_expect_from_vec(vec, q):
    prob = np.abs(vec)**2
    bits = (np.arange(prob.size) >> q) & 1   # qubit 0 = LSB
    return float(np.sum((1 - 2*bits) * prob))

def site_occupations(vec, nq):
    occs = []
    for i in range(nq // 2):
        q_up, q_dn = 2*i, 2*i+1
        n_up = 0.5*(1 - z_expect_from_vec(vec, q_up))
        n_dn = 0.5*(1 - z_expect_from_vec(vec, q_dn))
        occs.append(n_up + n_dn)
    return occs

occs = site_occupations(svT.data, nq)

print("Site occupations n_i (Qiskit exact):")
for i, ni in enumerate(occs):
    print(f"  i={i}: n_i={ni:.6f}")

Site occupations n_i (Qiskit exact):
  i=0: n_i=1.930536
  i=1: n_i=1.912186
  i=2: n_i=0.089344
  i=3: n_i=0.067935
